In [ ]:
from pydantic import BaseModel, ConfigDict, ValidationError, model_validator, BeforeValidator, AfterValidator
from numpydantic import NDArray, Shape
from typing import Optional, Any, Annotated
import numpy as np
from pchandler.v2.base_arrays import BaseArray, AffineArray, RotationArray
from pchandler.v2.geometry.coordinates import CartesianCoordinates, SphericalCoordinates

In [ ]:
array_N_T = NDArray[Shape["*"],Any]
array_N_3_T = NDArray[Shape["*, 3"],Any]
array_N_3_u8 = NDArray[Shape["*, 3"], np.uint8]
array_4_4_T = NDArray[Shape["4, 4"],Any]

In [ ]:
n = 100000
coords = BaseArray(arr=np.random.rand(n,3))
rand = np.random.rand(n,4)

#### Overhead test on the model copy + validate method

In [ ]:
%%timeit -n 100
BaseArray.model_validate(coords.model_copy(update={'xyz': rand}))

#### Overhead test on the model_dump and re-initialise approach

In [ ]:
%%timeit -n 100
b = BaseArray(**(coords.model_dump()| {'xyz': rand}))

### Tests comparing the overhead on functions working with the object vs numpy array directly

In [ ]:
n = 10000000
xyz = np.random.rand(n,3)
coords = BaseArray(arr=xyz.copy())

In [ ]:
%%timeit -n 50
c = coords + 1

In [ ]:
%%timeit -n 50

b = xyz + 1

In [ ]:
c = coords + 1
isinstance(c, BaseArray)

In [ ]:
from __future__ import annotations
from collections import OrderedDict
import copy
from typing import Any



class TransformRecord(OrderedDict):
    def __int__(self):
        super(TransformRecord, self).__init__()

    def __setitem__(self, key, value):
        if isinstance(value, (Array4x4, np.ndarray)):
            return super(TransformRecord, self).__setitem__(key, value)
        raise ValueError(f'Cannot set transform record value of type: {type(value)}')

    def rollback_record(self, index: int) -> tuple[Any, TransformRecord]:
        previous_history = type(self)()
        remaining_transforms = copy.deepcopy(self)

        for i in range(0, index):
            item = remaining_transforms.popitem(last=False)
            previous_history[item[0]] = item[1]

        chained_transform = remaining_transforms.popitem()[1]
        for t in remaining_transforms.items():
            chained_transform @= t[1]

        return chained_transform, previous_history

from scipy.spatial.transform import Rotation

translate = lambda x: np.array([[1, 0, 0, x[0]],
                                [0, 1, 0, x[1]],
                                [0, 0, 1, x[2]],
                                [0, 0, 0, 1]])

forward = translate(np.ones(3)*2)
backward = translate(np.ones(3)*-4)

# --------------------------------------------------
# In case a project transformation is passed
a = TransformRecord()
a['PRJ_0'] = INPUT_TRANSFORMATION
a['SOC_1'] = np.eye(4)  # then check if global shift needed ... "shouldn't" be the case...
a['REG_2'] = CustomTransformFromRegistration

# --------------------------------------------------
# No Transformation Passed
a = TransformRecord()
a['SOC_0'] = np.eye(4)
a['OPT_1'] = np.eye(4)[:3, :] - global_shift    # Where global shift needed
a['REG_2'] = CustomTransformFromRegistration    # Global shift checked
a['OPT_3'] = np.eye(4)[:3, :] - new_global_shift
a['TRANSLATED_4'] = np.eye(4)[:3, :] - arbitrary_translation

transform, old_transforms = a.rollback_record(2)

In [ ]:
print(transform)

In [ ]:
print(old_transforms)

In [1]:
import numpy as np
import timeit

# Size of test data
shape = (1000000, 3)
data = np.random.rand(*shape)

# Class using __array__
class ArrayWrapper:
    def __init__(self, arr):
        self.arr = arr

    def __array__(self, dtype=None):
        return np.array(self.arr, dtype=dtype)

# Class using __array_interface__
class InterfaceWrapper:
    def __init__(self, arr):
        self._arr = arr

    @property
    def __array_interface__(self):
        return self._arr.__array_interface__


# Create instances
array_wrapper = ArrayWrapper(data)
interface_wrapper = InterfaceWrapper(data)

# Define timing code
setup_code = "from __main__ import np, array_wrapper, interface_wrapper"

# Measure time to convert using np.array (will call __array__)
# time_array_wrapper = timeit.timeit("np.array(array_wrapper)", setup=setup_code, number=100)
#
# # Measure time to convert using np.asarray (should use __array_interface__ if defined)
# time_interface_wrapper = timeit.timeit("np.asarray(interface_wrapper)", setup=setup_code, number=100)

# (time_array_wrapper, time_interface_wrapper)

In [2]:
# Measure time to convert using np.array (will call __array__)
time_array_wrapper = timeit.timeit("np.linalg.norm(np.array(array_wrapper))", setup=setup_code, number=100)
# Measure time to convert using np.asarray (should use __array_interface__ if defined)
time_interface_wrapper = timeit.timeit("np.linalg.norm(np.asarray(interface_wrapper))", setup=setup_code, number=100)
(time_array_wrapper, time_interface_wrapper)

(1.2966715000000022, 0.039952900000002955)

In [25]:
from pchandler.v2.base_arrays import BaseArray
from pchandler.v2.geometry.core import PointCloudData
from pchandler.v2.geometry.core_pydantic import BasePointCloud
from pchandler.v2.geometry.scalar_fields_pydantic import ScalarFieldManager

import timeit
data = np.random.rand(1000000, 3)
time_point_cloud_data = timeit.timeit("PointCloudData(xyz=data)", setup=setup_code, number=100, globals=globals())
time_base_array = timeit.timeit("BaseArray(arr=data)", setup=setup_code, number=100, globals=globals())

time_base_pcd = timeit.timeit("BasePointCloud(arr=data, sfm=ScalarFieldManager())", setup=setup_code, number=100, globals=globals())

(time_point_cloud_data, time_base_array, time_base_pcd)

(1.5332037000002856, 0.002092500000344444, 0.0025654999999460415)

In [26]:
a = PointCloudData(xyz=data)
b = BaseArray(arr=data.astype(np.float32))
c = BasePointCloud(arr=data.astype(np.float32), sfm=ScalarFieldManager())

In [28]:
%%timeit -r 5 -n 100
a = PointCloudData(xyz=data)

16.1 ms ± 794 μs per loop (mean ± std. dev. of 5 runs, 100 loops each)


Point Cloud Initialisation - 14.4 ms ± 1.99 ms per loop (mean ± std. dev. of 5 runs, 100 loops each)

In [36]:
%%timeit -r 5 -n 100
a_ = np.linalg.norm(a.xyz) * 2.3 + 3.2 - 1.22 * 2323
a__ = np.sin(a.xyz)

7.69 ms ± 149 μs per loop (mean ± std. dev. of 5 runs, 100 loops each)


Point Cloud Computation - 7.33 ms ± 436 μs per loop (mean ± std. dev. of 5 runs, 1,000 loops each)

In [37]:
%%timeit -r 5 -n 100
b = BaseArray(arr=data.astype(np.float32))

4.63 ms ± 102 μs per loop (mean ± std. dev. of 5 runs, 100 loops each)


Base Array Initialisation - 4.8 ms ± 286 μs per loop (mean ± std. dev. of 5 runs, 1,000 loops each)

In [38]:
%%timeit -r 5 -n 100
b_ = np.linalg.norm(b.arr) * 2.3 + 3.2 - 1.22 *2323
b__ = np.sin(b.arr)

7.64 ms ± 156 μs per loop (mean ± std. dev. of 5 runs, 100 loops each)


Base Array Computation - 7.17 ms ± 238 μs per loop (mean ± std. dev. of 5 runs, 1,000 loops each)7.38 ms ± 263 μs per loop (mean ± std. dev. of 5 runs, 1,000 loops each)

In [39]:
%%timeit -r 5 -n 100
c = BasePointCloud(arr=data.astype(np.float32), sfm=ScalarFieldManager())

4.61 ms ± 160 μs per loop (mean ± std. dev. of 5 runs, 100 loops each)


In [40]:
%%timeit -r 5 -n 100
c_ = np.linalg.norm(c.arr) * 2.3 + 3.2 - 1.22 *2323
f = np.sin(c.arr)

7.51 ms ± 141 μs per loop (mean ± std. dev. of 5 runs, 100 loops each)


In [12]:
ind = np.zeros(b.shape[0], dtype=np.bool_)
ind[np.random.randint(0, b.shape[0], 10000)] = True
np.sum(ind)

np.int64(9947)

In [13]:
%%timeit -r 5 -n 100
mask = a._convert_indexing_to_mask(ind)
da = a.sample(ind)

2.14 ms ± 19.2 μs per loop (mean ± std. dev. of 5 runs, 100 loops each)


2.12 ms ± 16.9 μs per loop (mean ± std. dev. of 5 runs, 1,000 loops each)

In [14]:
%%timeit -r 5 -n 100
db = b.sample(ind)

3.88 ms ± 954 μs per loop (mean ± std. dev. of 5 runs, 100 loops each)


3.38 ms ± 53.3 μs per loop (mean ± std. dev. of 5 runs, 1,000 loops each)

In [10]:
%%timeit -r 5 -n 100
a = b[ind]
b = b[~ind]

UnboundLocalError: cannot access local variable 'b' where it is not associated with a value

In [13]:
%%timeit -r 5 -n 100
a, c = b[ind], b[~ind]

IndexError: too many indices for array